# Datekcja pierwszoplanowych obiektów ruchomych

Sekwencje obrazów to klatki wideo wyciągnięte do osobnych plików .jpg. ```groundtruth``` to klucz odpowiedzi o okerślonych wartościach:
- 0(czarny) - Tło (np. ulica)
- 50(ciemnoszary) - Cień rzucany przez obiekt
- 85(szary) - poza obszarem zainteresowania
- 170(jasnoszary) - nieznany ruch
- 255(biały) - obiekt pierwszoplanowy (tego szukamy)

In [1]:
import os
import cv2
import glob
import numpy as np

In [3]:
def process_sequence(dataset_path, threshold_value):

	input_path = os.path.join(dataset_path, "input", "*.jpg")
	gt_path = os.path.join(dataset_path, "groundtruth", "*.png")
	
	input_files = sorted(glob.glob(input_path))
	gt_files = sorted(glob.glob(gt_path))

	if not input_files or not gt_files:
		print("Nie znaleziono plików w katalogu.")
		return
	
	roi_file = os.path.join(dataset_path, "temporalROI.txt")
	if os.path.exists(roi_file):
		with open(roi_file, "r") as f:
			start_frame, end_frame = map(int, f.read().split())
	else:
		#analizowane klatki od początku do końca
		start_frame, end_frame = 1, len(input_files)
	print(f"Analizowane klatki to {start_frame} - {end_frame}")

	prev_frame = None

	for i in range(start_frame - 1, end_frame):
		img = cv2.imread(input_files[i])
		gt_mask = cv2.imread(gt_files[i], cv2.IMREAD_GRAYSCALE)

		if img is None or gt_mask is None:
			continue
	
		_, binary_gt = cv2.threshold(gt_mask, 254, 255, cv2.THRESH_BINARY)
		curr_frame_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

		if prev_frame is not None:
			diff = np.abs(curr_frame_gray.astype('int') - prev_frame.astype('int'))
			diff = diff.astype('uint8')
			_, binary_diff = cv2.threshold(diff, threshold_value, 255, cv2.THRESH_BINARY)

			cv2.imshow("Klatka Wyjsciowa", img)
			cv2.imshow("Oryginalna Maska", binary_gt)
			cv2.imshow("Roznica", binary_diff)
		
		prev_frame = curr_frame_gray.copy()

		if cv2.waitKey(10) & 0xFF == 27:
			break
	cv2.destroyAllWindows()


process_sequence("highway", 15)


Analizowane klatki to 470 - 1700


### Wersja z operacjami morfologicznymi

In [21]:
def process_sequence_morph(dataset_path, threshold_value):

	input_path = os.path.join(dataset_path, "input", "*.jpg")
	gt_path = os.path.join(dataset_path, "groundtruth", "*.png")
	
	input_files = sorted(glob.glob(input_path))
	gt_files = sorted(glob.glob(gt_path))

	if not input_files or not gt_files:
		print("Nie znaleziono plików w katalogu.")
		return
	
	roi_file = os.path.join(dataset_path, "temporalROI.txt")
	if os.path.exists(roi_file):
		with open(roi_file, "r") as f:
			start_frame, end_frame = map(int, f.read().split())
	else:
		start_frame, end_frame = 1, len(input_files)
	print(f"Analizowane klatki to {start_frame} - {end_frame}")

	prev_frame = None

	#Zmienne do liczenia statystyk
	total_TP, total_FP, total_FN = 0, 0, 0

	for i in range(start_frame - 1, end_frame):
		img = cv2.imread(input_files[i])
		gt_mask = cv2.imread(gt_files[i], cv2.IMREAD_GRAYSCALE)

		if img is None or gt_mask is None:
			continue
	
		_, binary_gt = cv2.threshold(gt_mask, 254, 255, cv2.THRESH_BINARY)
		curr_frame_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

		if prev_frame is not None:
			diff = np.abs(curr_frame_gray.astype('int') - prev_frame.astype('int'))
			diff = diff.astype('uint8')
			_, binary_diff = cv2.threshold(diff, threshold_value, 255, cv2.THRESH_BINARY)

			blurred_diff = cv2.medianBlur(binary_diff,7)
			morph_diff = cv2.morphologyEx(blurred_diff, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8), iterations=3)

			img_vis = img.copy()

			retval, labels, stats, centroids = cv2.connectedComponentsWithStats(morph_diff)

			
			if stats.shape[0] > 1:
				areas = stats[1:, 4]
				largest_idx = np.argmax(areas)
				pi = largest_idx + 1
				x, y, w, h, area = stats[pi]
				cx, cy = centroids[pi]
				cv2.rectangle(img_vis, (x, y), (x + w, y + h), (255, 0, 0), 2)
				cv2.putText(img_vis, f"Area: {area}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)


			# Ewaluacja wyników
			TP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 255))
			FP = np.sum(np.logical_and(morph_diff == 255, binary_gt == 0))
			FN = np.sum(np.logical_and(morph_diff == 0, binary_gt == 255))

			total_TP += TP
			total_FP += FP
			total_FN += FN
			
			if(retval > 1):
				labels_vis = np.uint8((labels / retval) * 255)
				cv2.imshow("Wizualizacja etykiet", labels_vis)

			cv2.imshow("Wynik po labelowaniu", img_vis)
			cv2.imshow("Klatka Wyjsciowa", img)
			cv2.imshow("Oryginalna Maska", binary_gt)
			cv2.imshow("Po filtracji medianowej", blurred_diff)
			cv2.imshow("Roznica", binary_diff)
			cv2.imshow("Po operacjach morfologicznych", morph_diff)
		
		prev_frame = curr_frame_gray.copy()

		if cv2.waitKey(10) & 0xFF == 27:
			break
	cv2.destroyAllWindows()

	print("=== PODSUMOWANIE WYNIKÓW ===")

	if(total_TP + total_FP) > 0:
		precision = total_TP / (total_TP + total_FP)
	
	if(total_TP + total_FN) > 0:
		recall = total_TP / (total_TP + total_FN)

	if(precision + recall) > 0:
		f1_score = 2 * (precision * recall) / (precision + recall)

	print(f"Precision: {precision:.4f}")
	print(f"Recall: {recall:.4f}")
	print(f"F1-Score: {f1_score:.4f}")


datasets = ['highway', 'office', 'pedestrian']
for d in datasets:
	process_sequence_morph(d, 15)


Analizowane klatki to 470 - 1700
=== PODSUMOWANIE WYNIKÓW ===
Precision: 0.7707
Recall: 0.7860
F1-Score: 0.7783
Analizowane klatki to 570 - 2050
=== PODSUMOWANIE WYNIKÓW ===
Precision: 0.7751
Recall: 0.0591
F1-Score: 0.1098
Analizowane klatki to 300 - 1099
=== PODSUMOWANIE WYNIKÓW ===
Precision: 0.4845
Recall: 0.7947
F1-Score: 0.6020


# Podsumowanie wyników

### Dataset Highway
- Precision $\approx$ 77%
- Recall $\approx$ 79%
- F1-Score $\approx$ 78%

Wszystkie metryki są zadowalające. Autostrada to idealne środowisko do odejmowania klatek, ponieważ samochody poruszają się szybko i są duże. Algorytm z łatwością wykrywa duże "plamy" ruchu.

### Office Highway
- Precision $\approx$ 78%
- Recall $\approx$ 6%
- F1-Score $\approx$ 10%

Na sekwencji biurowej jest jedna osoba, która porusza się bardzo powoli lub wcale. Precyzja pozostaje wysoka gdyż jeśli już ruch zaistniał to jest zauważany. Niska czułość świadczy o tym, że jest dużo klatek, w których człowiek był w pokoju lecz się nie poruszał (maska go zawierała a różnica nie).

### Pedestrian Highway
- Precision $\approx$ 48%
- Recall $\approx$ 79%
- F1-Score $\approx$ 60%

Algorytm wyłapuje bardzo dużo FP. Dzieje się tak, ponieważ piesi poruszają się wolno a dodatkowo przy obecnym na obrazach słońcu zostawiają cienie rzucane na chodnik. To one obniżają jakość algorytmu.